#Activity 2: Building an MLP with Keras — Solving XOR

In [1]:
!pip install numpy keras tensorflow

##1. Prepare the XOR data

In [2]:
import numpy as np
import keras
from keras import layers

print("Keras version:", keras.__version__)
print("Keras backend:", keras.config.backend())

# Four rows of the XOR truth table.
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=np.float32)

y = np.array([
    [0],
    [1],
    [1],
    [0]
], dtype=np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Inputs:\n", X)
print("Labels:\n", y)

Keras version: 3.13.2
Keras backend: tensorflow
X shape: (4, 2)
y shape: (4, 1)
Inputs:
 [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
Labels:
 [[0.]
 [1.]
 [1.]
 [0.]]


###Your tasks

- X contains 4 examples, each with 2 input features (x1, x2).
- y contains 4 labels, one binary label per example.
- We want to predict one of two classes: 0 (inputs are equal) or 1 (inputs differ). We are not predicting an unrestricted numerical quantity.
    
- The correct input/output pairs are (0,0) → 0, (0,1) → 1, (1,0) → 1, and (1,1) → 0.

##2. Build the MLP

### Before Running:

The hidden layer is created by:

      model.add(layers.Dense(units=4, activation="relu", name="hidden_layer"))

There is only one output neuron because XOR is a binary classification problem. The sigmoid output produces a single score between 0 and 1 that represents the probability of class 1.

In [3]:
# Use a fixed seed so repeated runs in a similar environment
# are easier to compare. A seed does NOT guarantee 100% accuracy.
import keras
from keras import layers
seed = 32
keras.utils.set_random_seed(seed)

model = keras.Sequential(name="MLP_XOR_Solver")
model.add(keras.Input(shape=(2,)))
model.add(layers.Dense(units=4, activation="relu", name="hidden_layer"))
model.add(layers.Dense(units=1, activation="sigmoid", name="output_layer"))

model.summary()

Model: "MLP_XOR_Solver"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer (Dense)            │ (None, 4)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17 (68.00 B)

 Trainable params: 17 (68.00 B)

 Non-trainable params: 0 (0.00 B)

###Your tasks
- `keras.Input(shape=(2,))` describes two input features; it does **not** add trainable parameters.
- `Dense(units=4, activation="relu")` is the hidden layer.
- `Dense(units=1, activation="sigmoid")` is the output layer. One output score is sufficient for a binary classification task.
- A nonlinear activation is essential here: without nonlinear hidden activations, a stack of `Dense` layers would still be equivalent to a single linear transformation, which cannot represent XOR.

| Layer | Weights | Biases | Parameters |
|---|---:|---:|---:|
| Hidden: 2 inputs → 4 neurons | `2 × 4 = 8` | `4` | **12** |
| Output: 4 inputs → 1 neuron | `4 × 1 = 4` | `1` | **5** |
| **Total** | **12** | **5** | **17** |

In symbols, the network first calculates hidden activations and then an output score: $`h = \mathrm{ReLU}(xW_1 + b_1), \qquad \hat{y} = \sigma(hW_2 + b_2)`$

<!-- $$
h = \mathrm{ReLU}(xW_1 + b_1), \qquad \hat{y} = \sigma(hW_2 + b_2)
$$ -->

Here, $\sigma$ is sigmoid. You **do not** need to calculate the matrix multiplications by hand in this activity.

##3. Compile the model

In [4]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5)]
)

###Your tasks

1. `loss="binary_crossentropy"` specifies the loss that training aims to reduce.
2. `keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5)` specifies the reported accuracy. Using an explicit metric also makes the classification threshold clear.
3. Each target is one of two classes, `0` and `1`. The metric converts sigmoid scores into binary predictions and checks them against the labels.

The **optimizer** updates parameters using information from the loss. The **accuracy metric** helps us interpret results; it is **not** the quantity Adam directly minimizes here.

We will study the mathematics of optimization later.

##4. Train the MLP

In [5]:
num_epochs = 1000

history = model.fit(
    X,
    y,
    epochs=num_epochs,
    batch_size=4,
    shuffle=False,
    verbose=0
)

# Evaluate the final trained model on the four training examples.
train_loss, train_accuracy = model.evaluate(X, y, verbose=0)

print(f"Final training loss: {train_loss:.4f}")
print(f"Final training accuracy: {train_accuracy:.0%}")

Final training loss: 0.4603
Final training accuracy: 75%


###Your tasks

1. One **epoch** is one pass through the four training examples. Because `batch_size=4`, the entire XOR dataset is used together in each training update (**full-batch training**).
2. **75%** means **3 of the 4** XOR examples were correctly classified. **50%** means 2 out of 4.
3. **No.** Weights start from an initialization, and training may fail to reach a correct solution within the selected number of epochs. A disappointing result does **not necessarily** mean the code is wrong, and it does not mean the architecture is incapable of representing XOR.
4. We used the **same four examples for fitting and evaluation**. Nothing has been reserved as unseen test data.

**Key distinction:**

- **Representation:** Does there exist a set of weights that makes the network solve XOR? **Yes.**
- **Training/optimization:** Does this particular training run find such weights? **Not always.**

The seed helps make classroom results more repeatable in similar environments, but hardware and software differences can still affect training.

##5. Inspect each prediction

In [6]:
scores = model.predict(X, verbose=0).reshape(-1)

# Use the same decision rule as BinaryAccuracy with threshold=0.5.
predictions = (scores > 0.5).astype(int)

for inputs, score, predicted, expected in zip(
    X, scores, predictions, y.reshape(-1)
):
    print(
        f"Input: {inputs.astype(int)} | "
        f"Score: {score:.3f} | "
        f"Predicted: {predicted} | Expected: {int(expected)}"
    )

calculated_accuracy = np.mean(predictions == y.reshape(-1))
print(f"Accuracy calculated from predictions: {calculated_accuracy:.0%}")

Input: [0 0] | Score: 0.522 | Predicted: 1 | Expected: 0
Input: [0 1] | Score: 0.806 | Predicted: 1 | Expected: 1
Input: [1 0] | Score: 0.522 | Predicted: 1 | Expected: 1
Input: [1 1] | Score: 0.211 | Predicted: 0 | Expected: 0
Accuracy calculated from predictions: 75%


###Your tasks

1. **It depends on your run.** Compare each printed prediction with its expected value. If training succeeded perfectly, all four pairs match: `0, 1, 1, 0`.
2. Sigmoid produces a continuous score between 0 and 1. We then apply a threshold to choose class `0` or `1`. Such a score is often interpreted as a model-estimated probability, but it is not automatically guaranteed to be well calibrated.
3. A score **greater than 0.5** becomes class `1`; a score at or below 0.5 becomes class `0`. This is the convention used by the explicit `BinaryAccuracy` metric in Section 3.
4. **Yes**, allowing for rounding of displayed percentages. Both use the same four examples and the same threshold.
5. One step-function perceptron has one linear decision boundary and cannot represent XOR. An MLP has a hidden layer with nonlinear activations and **can** represent XOR, although one training run may still fail.

A possible **successful** run might show low scores for `[0,0]` and `[1,1]`, and high scores for `[0,1]` and `[1,0]`. **The exact decimal scores are not fixed expected output.**

##6. Experiment: change one setting

### Your tasks
There is **no single correct results table**: model outcomes can vary with initialization and environment. Report actual measurements, not a prescribed 100% score.

Possible interpretations:

- **2 hidden units:** the network still has enough representational capacity for XOR, but training can be sensitive to the starting weights.
- **8 hidden units:** the model has more trainable parameters and might learn XOR more reliably in some trials; **it is not guaranteed** to be faster or better in every run.
- **`tanh` instead of `relu`:** both are nonlinear and can represent XOR. Results may differ because the activations behave differently; neither is universally superior.
- **2000 instead of 1000 epochs:** extra training steps sometimes reduce loss or change predictions, but they do not guarantee success and cannot necessarily fix ineffective optimization.

Parameter counts if you choose the capacity experiment:

| Architecture | Trainable parameters |
|---|---:|
| 2 inputs → **2** hidden → 1 output | `(2 × 2 + 2) + (2 × 1 + 1) = 9` |
| 2 inputs → **4** hidden → 1 output | **17** |
| 2 inputs → **8** hidden → 1 output | `(2 × 8 + 8) + (8 × 1 + 1) = 33` |


In [7]:
def run_trial(hidden_units=4, activation="relu", epochs=1000, seed=32):
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)

    candidate = keras.Sequential([
        keras.Input(shape=(2,)),
        layers.Dense(hidden_units, activation=activation),
        layers.Dense(1, activation="sigmoid")
    ])
    candidate.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5)]
    )
    candidate.fit(X, y, epochs=epochs, batch_size=4,
                  shuffle=False, verbose=0)
    loss, accuracy = candidate.evaluate(X, y, verbose=0)
    return accuracy, loss

for trial_seed in [7, 21, 32]:
    accuracy, loss = run_trial(hidden_units=4, seed=trial_seed)
    print(f"Seed {trial_seed}: accuracy={accuracy:.0%}, loss={loss:.4f}")

Seed 7: accuracy=100%, loss=0.2992
Seed 21: accuracy=100%, loss=0.3975
Seed 32: accuracy=75%, loss=0.4603


##7. Troubleshooting and concept check


**1. Not necessarily.** A score of 75% means 3 out of 4 labels were correctly predicted. First check the data, model structure, compilation, and classification threshold. If these are correct, the run may simply have learned an imperfect solution.

**2.** A suitable combination of weights **exists**, but the training process begins from particular initial weights and has a limited number of updates. Being able to represent a function is different from successfully learning its weights.

**3. Try one change at a time:**

- Rebuild the model and try a different random seed.
- Try `tanh` in the hidden layer.
- Try more hidden units.
- Try more epochs, especially if loss is still decreasing.

**Do not automatically conclude** that any one of these changes will solve the problem. Compare the actual predictions and loss.

**Optional background (not required for this lesson):** With ReLU, a hidden unit whose pre-activation is nonpositive on every training example may receive no useful gradient signal from those examples. This is one possible reason that some initializations learn poorly. It does **not** follow that every run below 100% is caused by a “dead ReLU,” nor that adding epochs can always resolve the problem. The details belong in a later lesson about optimization and backpropagation.

##8. Wrap-up

The MLP contains a hidden layer with nonlinear activation functions, allowing it to represent nonlinear relationships such as XOR. A single perceptron cannot.